# 02 — Бейзлайн

Насколько хорошо можно предсказать `damage_rel` без умной модели.

| Уровень | Предсказание | Что знает |
| --- | --- | --- |
| 0. Константа | всегда 1.0 | ничего |
| 1. Тепловая карта | средний `damage_rel` в зоне | карта, сторона, зона |
| 1б. Сглаженная карта | то же, но редкие зоны тянутся к 1.0 | карта, сторона, зона |

Модель (следующий ноутбук) должна обогнать лучший из них.

## 1. Загрузка

In [98]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

features = pd.read_parquet("../data/processed/features.parquet")
print(features.shape, "боёв:", features["arena_id"].nunique())

(57959, 45) боёв: 4075


## 2. Заморозить обучающие бои

Финальная проверка будет на **новых** реплеях: всё, что есть сейчас, — для обучения и подбора параметров; всё, что вы скачаете потом, — только для проверки (раздел 11).

Список текущих боёв сохраняется **один раз**. Ячейка не перезапишет файл, если он уже есть: иначе после докачки новые бои попали бы в «старые».

`dev` — только замороженные бои. Разделы 3–10 работают с ним, поэтому новые бои никак не влияют на выбор параметров.

In [99]:
from pathlib import Path

FROZEN = Path("../data/processed/train_battles.csv")

if FROZEN.exists():
    print("список уже заморожен:", FROZEN)
else:
    pd.DataFrame({"arena_id": features["arena_id"].unique()}).to_csv(FROZEN, index=False)
    print("заморожено боёв:", features["arena_id"].nunique())

dev = features[features["arena_id"].isin(pd.read_csv(FROZEN)["arena_id"])]
print("для обучения и подбора параметров:", dev["arena_id"].nunique(), "боёв")

список уже заморожен: ..\data\processed\train_battles.csv
для обучения и подбора параметров: 3485 боёв


## 3. Разделение на обучение и проверку

Делим **по боям**: все союзники одного боя целиком идут либо в обучение, либо в проверку.
Иначе модель «подглядит» в ответ: среднее `damage_rel` в бою по определению равно 1.

`rng` с зерном 42 — чтобы при каждом запуске разделение было одинаковым.

In [100]:
TEST_SHARE = 0.2

battles = dev["arena_id"].unique()
rng = np.random.default_rng(42)
test_ids = rng.choice(battles, size=int(len(battles) * TEST_SHARE), replace=False)

is_test = dev["arena_id"].isin(test_ids)
train = dev[~is_test]
test = dev[is_test]

print("обучение:", len(train), "строк,", train["arena_id"].nunique(), "боёв")
print("проверка:", len(test), "строк,", test["arena_id"].nunique(), "боёв")
print("общих боёв:", len(set(train["arena_id"]) & set(test["arena_id"])))

обучение: 39762 строк, 2788 боёв
проверка: 9857 строк, 697 боёв
общих боёв: 0


## 4. Метрика

**MAE** — средняя абсолютная ошибка: |предсказание − правда|, усреднённая по строкам. Меньше — лучше.

In [101]:
def mae(prediction, truth):
    return float(np.mean(np.abs(np.asarray(prediction) - np.asarray(truth))))


results = {}

## 5. Уровень 0: константа

In [102]:
results["0. константа 1.0"] = mae(np.ones(len(test)), test["damage_rel"])
results

{'0. константа 1.0': 0.662041797201}

## 6. Уровень 1: тепловая карта

Для каждой тройки (карта, сторона, зона) считаем на **обучении** средний `damage_rel` и сколько там было танков.
Для проверочной строки берём среднее её зоны.

Тонкость: в зоне, где в обучении было 2–3 танка, среднее — почти случайное число.
Поэтому если танков в зоне меньше `min_count`, предсказываем 1.0, как константа.

In [103]:
KEYS = ["map_name", "team", "zone"]

zone_stats = train.groupby(KEYS)["damage_rel"].agg(zone_mean="mean", zone_count="count").reset_index()
zone_stats.sort_values("zone_count", ascending=False).head()

,map_name,team,zone,zone_mean,zone_count
347,05_prohorovka,1,-4_2,1.256632,125
661,08_ruinberg,1,-4_0,1.510591,100
2133,18_cliff,2,0_0,1.179937,99
403,05_prohorovka,2,-4_-4,1.056493,96
938,10_hills,1,-1_-1,1.089000,95


In [104]:
def heatmap_predict(test, zone_stats, min_count):
    """Средний damage_rel зоны из обучения; 1.0, если зона новая или в ней мало танков."""
    merged = test[KEYS].merge(zone_stats, on=KEYS, how="left")
    reliable = merged["zone_count"] >= min_count
    return np.where(reliable, merged["zone_mean"], 1.0)


for min_count in [1, 5, 10, 20, 50, 100, 200]:
    pred = heatmap_predict(test, zone_stats, min_count)
    print(f"min_count={min_count:>3}: MAE = {mae(pred, test['damage_rel']):.4f}")

min_count=  1: MAE = 0.6882
min_count=  5: MAE = 0.6760
min_count= 10: MAE = 0.6766
min_count= 20: MAE = 0.6724
min_count= 50: MAE = 0.6654
min_count=100: MAE = 0.6628
min_count=200: MAE = 0.6620


## 7. Уровень 1б: сглаженная тепловая карта

Вместо жёсткого порога — плавный: среднее зоны «притягивается» к 1.0 тем сильнее, чем меньше в ней танков.

```
сглаженное = (сумма damage_rel в зоне + k × 1.0) / (число танков + k)
```

Если в зоне 3 танка, а k = 20, ответ почти 1.0. Если 300 танков — почти чистое среднее зоны.
Это называется **сглаживание** (shrinkage) — стандартный приём, когда групп много, а данных в части из них мало.

In [105]:
def smoothed_predict(test, train, k, keys=KEYS):
    stats = train.groupby(keys)["damage_rel"].agg(zone_sum="sum", zone_count="count").reset_index()
    stats["smoothed"] = (stats["zone_sum"] + k * 1.0) / (stats["zone_count"] + k)
    merged = test[keys].merge(stats, on=keys, how="left")
    return merged["smoothed"].fillna(1.0).to_numpy()


for k in [1, 5, 10, 20, 50, 100, 200, 500]:
    pred = smoothed_predict(test, train, k)
    print(f"k={k:>3}: MAE = {mae(pred, test['damage_rel']):.4f}")

k=  1: MAE = 0.6766
k=  5: MAE = 0.6669
k= 10: MAE = 0.6639
k= 20: MAE = 0.6617
k= 50: MAE = 0.6602
k=100: MAE = 0.6602
k=200: MAE = 0.6606
k=500: MAE = 0.6612


## 8. Итог по MAE

Выберите лучший `min_count` и лучший `k` из таблиц выше и подставьте сюда.

In [106]:
BEST_MIN_COUNT = 10   # поменяйте на лучший из раздела 6
BEST_K = 50           # поменяйте на лучший из раздела 7

results[f"1. тепловая карта, min_count={BEST_MIN_COUNT}"] = mae(heatmap_predict(test, zone_stats, BEST_MIN_COUNT), test["damage_rel"])
results[f"1б. сглаженная карта, k={BEST_K}"] = mae(smoothed_predict(test, train, BEST_K), test["damage_rel"])

pd.Series(results).sort_values()

1б. сглаженная карта, k=50         0.660197
0. константа 1.0                   0.662042
1. тепловая карта, min_count=10    0.676617
dtype: float64

## 9. Главная проверка: помогает ли совет «топ-3 зоны»

MAE отвечает на вопрос «насколько точно предсказан урон». А продукт отвечает на другой: **если поехать в одну из 3 советуемых зон, урона будет больше?**

1. На **обучении** для каждой пары (карта, сторона) выбираем 3 зоны с наибольшим сглаженным `damage_rel` (среди зон, где было хотя бы `min_count` танков).
2. На **проверке** смотрим: у танков, которые на 90-й секунде оказались в одной из этих зон, `damage_rel` в среднем выше, чем у остальных?

Это честный тест: зоны выбраны по старым боям, а проверяются на боях, которые карта не видела.

In [107]:
def top_zones(train, k, min_count, n=3, keys=KEYS):
    stats = train.groupby(keys)["damage_rel"].agg(zone_sum="sum", zone_count="count").reset_index()
    stats = stats[stats["zone_count"] >= min_count]
    stats["smoothed"] = (stats["zone_sum"] + k * 1.0) / (stats["zone_count"] + k)
    return (
        stats.sort_values("smoothed", ascending=False)
        .groupby([key for key in keys if key != "zone"])
        .head(n)[keys]
        .assign(in_top=True)
    )


top = top_zones(train, k=BEST_K, min_count=BEST_MIN_COUNT)
checked = test.merge(top, on=KEYS, how="left")
checked["in_top"] = checked["in_top"].fillna(False).astype(bool)

summary = checked.groupby("in_top")["damage_rel"].agg(["mean", "count"])
summary.index = summary.index.map({True: "в топ-3 зонах", False: "в остальных"})
summary

,mean,count
in_top,,
в остальных,0.968275,7726
в топ-3 зонах,1.299259,2131


In [108]:
lift = summary.loc["в топ-3 зонах", "mean"] / summary.loc["в остальных", "mean"]
print(f"Танки в советуемых зонах наносят в {lift:.2f} раза больше урона (относительно своей команды)")

Танки в советуемых зонах наносят в 1.34 раза больше урона (относительно своей команды)


## 10. То же по классам танков

Если «топ-3 зоны» хороши для одних классов и плохи для других — это аргумент, что модели нужен класс танка.

In [109]:
by_class = checked.groupby(["tank_class", "in_top"])["damage_rel"].mean().unstack()
by_class.columns = ["в остальных", "в топ-3"]
by_class["во сколько раз"] = by_class["в топ-3"] / by_class["в остальных"]
by_class.round(3)

,в остальных,в топ-3,во сколько раз
tank_class,,,
AT-SPG,1.028,1.429,1.391
SPG,0.835,0.232,0.278
heavyTank,1.029,1.309,1.272
lightTank,0.727,1.052,1.446
mediumTank,0.977,1.220,1.248


## 11. Проверка на новых боях

Запускать **после** того, как докачаете реплеи и заново выполните:

```
python -m spawn_advisor.dataset
python -m spawn_advisor.features
```

и перезапустите этот ноутбук (**Run All**).

Параметры `BEST_MIN_COUNT` и `BEST_K` выбраны в разделах 6–8 на старых боях, а здесь только применяются.
На новых боях **ничего не подбираем**: иначе проверка перестанет быть честной.

In [110]:
frozen_ids = pd.read_csv(FROZEN)["arena_id"]
old = features[features["arena_id"].isin(frozen_ids)]
new = features[~features["arena_id"].isin(frozen_ids)]
print("обучение (старые бои):", old["arena_id"].nunique(), "| проверка (новые бои):", new["arena_id"].nunique())

if len(new) == 0:
    print("Новых боёв пока нет: докачайте реплеи и пересоберите dataset и features.")
else:
    old_stats = old.groupby(KEYS)["damage_rel"].agg(zone_mean="mean", zone_count="count").reset_index()
    final = {
        "0. константа 1.0": mae(np.ones(len(new)), new["damage_rel"]),
        f"1. тепловая карта, min_count={BEST_MIN_COUNT}": mae(heatmap_predict(new, old_stats, BEST_MIN_COUNT), new["damage_rel"]),
        f"1б. сглаженная карта, k={BEST_K}": mae(smoothed_predict(new, old, BEST_K), new["damage_rel"]),
    }
    print(pd.Series(final).sort_values())

    top_new = top_zones(old, k=BEST_K, min_count=BEST_MIN_COUNT)
    checked_new = new.merge(top_new, on=KEYS, how="left")
    checked_new["in_top"] = checked_new["in_top"].fillna(False).astype(bool)
    summary_new = checked_new.groupby("in_top")["damage_rel"].agg(["mean", "count"])
    summary_new.index = summary_new.index.map({True: "в топ-3 зонах", False: "в остальных"})
    print(summary_new)

обучение (старые бои): 3485 | проверка (новые бои): 590


1б. сглаженная карта, k=50         0.676669
0. константа 1.0                   0.677810
1. тепловая карта, min_count=10    0.687687
dtype: float64
                   mean  count
in_top                        
в остальных    0.959820   6607
в топ-3 зонах  1.328017   1733


In [111]:
def lift(train, test, k, min_count, keys):
    top = top_zones(train, k=k, min_count=min_count, keys=keys)
    checked = test.merge(top, on=keys, how="left")
    checked["in_top"] = checked["in_top"].fillna(False).astype(bool)
    by_class = checked.groupby(["tank_class", "in_top"])["damage_rel"].mean().unstack().reindex(columns=[False, True])
    total = checked.loc[checked["in_top"], "damage_rel"].mean() / checked.loc[~checked["in_top"], "damage_rel"].mean()
    return total, by_class[True] / by_class[False]

In [112]:
KEYS_CLASS = ["map_name", "team", "tank_class", "zone"]

for keys, name in [(KEYS, "без класса"), (KEYS_CLASS, "по классам")]:
    for min_count in [10, 20, 50]:
        total, per_class = lift(train, test, k=BEST_K, min_count=min_count, keys=keys)
        print(f"{name}, min_count={min_count}: lift {total:.3f} |", per_class.round(2).to_dict())

без класса, min_count=10: lift 1.342 | {'AT-SPG': 1.39, 'SPG': 0.28, 'heavyTank': 1.27, 'lightTank': 1.45, 'mediumTank': 1.25}
без класса, min_count=20: lift 1.325 | {'AT-SPG': 1.44, 'SPG': 0.82, 'heavyTank': 1.21, 'lightTank': 1.3, 'mediumTank': 1.24}
без класса, min_count=50: lift 1.217 | {'AT-SPG': 1.13, 'SPG': 1.72, 'heavyTank': 1.19, 'lightTank': 0.85, 'mediumTank': 1.16}
по классам, min_count=10: lift 1.252 | {'AT-SPG': 1.08, 'SPG': 1.43, 'heavyTank': 1.28, 'lightTank': 1.04, 'mediumTank': 1.23}
по классам, min_count=20: lift 1.225 | {'AT-SPG': 1.05, 'SPG': 1.61, 'heavyTank': 1.2, 'lightTank': 0.89, 'mediumTank': 1.22}
по классам, min_count=50: lift 1.249 | {'AT-SPG': 0.85, 'SPG': nan, 'heavyTank': 1.21, 'lightTank': nan, 'mediumTank': 1.62}


In [113]:
N_FOLDS = 5


def cv_lift(data, k, min_count, keys, n_folds=N_FOLDS, seed=42):
    """Lift на n_folds разных разделениях по боям: общий и по классам, для каждого фолда."""
    battles = data["arena_id"].unique().copy()
    np.random.default_rng(seed).shuffle(battles)
    folds = np.array_split(battles, n_folds)

    totals, per_class = [], []
    for fold in folds:
        is_test = data["arena_id"].isin(fold)
        total, by_class = lift(data[~is_test], data[is_test], k, min_count, keys)
        totals.append(total)
        per_class.append(by_class)
    return pd.Series(totals), pd.concat(per_class, axis=1)


rows = []
for keys, name in [(KEYS, "без класса"), (KEYS_CLASS, "по классам")]:
    for min_count in [10, 20, 50]:
        totals, per_class = cv_lift(dev, k=BEST_K, min_count=min_count, keys=keys)
        row = {"вариант": name, "min_count": min_count,
               "lift": f"{totals.mean():.3f} ± {totals.std():.3f}"}
        for cls in per_class.index:
            values = per_class.loc[cls].dropna()
            row[cls] = f"{values.mean():.2f} ± {values.std():.2f}" if len(values) > 1 else "мало данных"
        rows.append(row)

pd.DataFrame(rows)

,вариант,min_count,lift,AT-SPG,SPG,heavyTank,lightTank,mediumTank
0,без класса,10,1.325 ± 0.018,1.30 ± 0.05,0.90 ± 0.35,1.29 ± 0.05,1.34 ± 0.10,1.26 ± 0.03
1,без класса,20,1.302 ± 0.023,1.28 ± 0.02,1.14 ± 0.18,1.25 ± 0.05,1.35 ± 0.07,1.21 ± 0.03
2,без класса,50,1.233 ± 0.034,1.19 ± 0.07,1.36 ± 0.64,1.18 ± 0.03,1.22 ± 0.10,1.15 ± 0.03
3,по классам,10,1.214 ± 0.027,1.11 ± 0.03,1.18 ± 0.02,1.28 ± 0.04,0.91 ± 0.10,1.12 ± 0.03
4,по классам,20,1.212 ± 0.012,1.08 ± 0.04,1.28 ± 0.09,1.21 ± 0.03,0.68 ± 0.12,1.10 ± 0.03
5,по классам,50,1.231 ± 0.084,1.02 ± 0.13,мало данных,1.19 ± 0.09,мало данных,1.11 ± 0.25
